# Machine learning: Random Forest and KSSHIBA (E. coli, Hospital A)

Same data, same folds and same reductions as the team's ML phase (Joud's notebook). Only the classifier changes, so the results can be compared directly with gradient boosting, SVM and MLP.

## Models

| Model | How it works  |
|---|---|
| Random Forest | Builds many decision trees, each on a random sample of isolates and bins. The trees vote. |
| KSSHIBA | Bayesian multi-view model (Guerrero-López et al., 2023). The spectra (as a linear kernel) and the R/S label share one latent space; the test labels are treated as missing and predicted. Authors' code: github.com/aguerrerolopez/RMPrediction |

**Inputs:** Random Forest uses the same four inputs as Joud (Kernel PCA RBF-200, Grouped ANOVA 300, LASSO auto C, all 6,000 bins). KSSHIBA uses all 6,000 bins only, because it performs its own reduction (kernel + latent space).

**Settings:** Random Forest default (300 trees), then tuned inside the training folds. KSSHIBA has no tuning step: its parameters are learned by maximising the ELBO (the authors' fixed settings, seed 0).

**Output:** `results_ml_member1.csv`, same columns as `results_ml_1.csv`: `antibiotic, input, model, auc, auc_std, prauc, n_features`.

## 1. Setup
Load the two shared files and the list of the 12 modelled antibiotics.
- `spectra_ecoli_A.npy`: 7,320 spectra × 6,000 bins
- `labels_ecoli_A.csv`: R = 1 / S = 0 / empty for each antibiotic, plus the fixed fold number (0–4, −1 = no label)

In [1]:
import numpy as np
import pandas as pd
import time, os, warnings
warnings.filterwarnings("ignore")
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.decomposition import KernelPCA
from sklearn.feature_selection import f_classif
from sklearn.model_selection import cross_val_score

X = np.load("shared_data/spectra_ecoli_A.npy")
lab = pd.read_csv("shared_data/labels_ecoli_A.csv")

MODEL_ABX = ["Ampicillin-Amoxicillin", "Cotrimoxazole", "Ciprofloxacin", "Levofloxacin",
             "Amoxicillin-Clavulanic acid", "Ceftriaxone", "Cefpodoxime", "Tobramycin",
             "Ceftazidime", "Norfloxacin", "Cefepime", "Piperacillin-Tazobactam"]
print(X.shape, lab.shape)   # expected (7320, 6000) (7320, 32)

(7320, 6000) (7320, 32)


## 2. The four inputs (reducers copied unchanged from Joud's notebook)

Each model is tested on the same four versions of the spectra. Every reducer is fitted on the **training folds only** (inside `run_cv_ml`), so the test fold never influences it.

| Input | Type | Features | Real bins? |
|---|---|---|---|
| Kernel PCA RBF-200 | Projection | 200 components | No (mixtures of all bins) |
| LASSO auto C | Embedded selection | about 220 bins | Yes |
| Grouped ANOVA 300 | Filter | 300 bins (30 windows of 10) | Yes |
| All 6,000 bins | No reduction | 6,000 bins | Yes |

In [2]:
class KernelPCAScaled(BaseEstimator, TransformerMixin):
    """Kernel PCA, then re-scale the components."""
    def __init__(self, n_components=200, kernel="rbf", gamma=None):
        self.n_components = n_components   # how many components to keep
        self.kernel = kernel               # "rbf" or "linear"
        self.gamma = gamma                 # None = sklearn default 1/n_features = 1/6000

    def fit(self, X, y=None):              # y is not used: kernel PCA is unsupervised
        self.kpca_ = KernelPCA(n_components=self.n_components, kernel=self.kernel,
                               gamma=self.gamma, eigen_solver="randomized", random_state=0)
        Z = self.kpca_.fit_transform(X)    # kernel matrix from TRAINING spectra only
        self.scaler_ = StandardScaler().fit(Z)
        return self

    def transform(self, X):                # test spectra are compared with the training spectra
        return self.scaler_.transform(self.kpca_.transform(X))

In [4]:
class LassoSelect(BaseEstimator, TransformerMixin):
    def __init__(self, C=0.03):          # C = penalty strength (small C = fewer bins kept)
        self.C = C

    def fit(self, X, y):
        m = LogisticRegression(penalty="l1", solver="liblinear", C=self.C, max_iter=1000).fit(X, y)
        w = np.abs(m.coef_[0])           # importance of each bin
        self.mask_ = w > 0               # keep only the bins with non-zero weight
        if not self.mask_.any():         # safety: if all were removed, keep 1 bin so nothing crashes
            self.mask_[0] = True
        self.weights_ = w                # saved, so we can plot the important regions later
        return self

    def transform(self, X):
        return X[:, self.mask_]


class LassoAutoC(BaseEstimator, TransformerMixin):
    def __init__(self, Cs=(0.01, 0.02, 0.03, 0.05, 0.1, 0.3), tol=0.005, cv=3):
        self.Cs = Cs          # C values to try (small C = fewer bins)
        self.tol = tol        # accept a smaller C if its AUC is within tol of the best
        self.cv = cv          # folds used inside the training data

    def fit(self, X, y):
        scores = []
        for C in self.Cs:
            m = LogisticRegression(penalty="l1", solver="liblinear", C=C, max_iter=1000)
            scores.append(cross_val_score(m, X, y, cv=self.cv, scoring="roc_auc").mean())
        scores = np.array(scores)
        ok = np.where(scores >= scores.max() - self.tol)[0]     # nearly-best choices
        self.C_ = self.Cs[ok[0]]                                # smallest C among them (Cs is sorted)
        m = LogisticRegression(penalty="l1", solver="liblinear", C=self.C_, max_iter=1000).fit(X, y)
        self.mask_ = np.abs(m.coef_[0]) > 0                     # keep bins with non-zero weight
        if not self.mask_.any():
            self.mask_[0] = True                                # safety: keep 1 bin
        return self

    def transform(self, X):
        return X[:, self.mask_]

In [5]:
class GroupedANOVASelector(BaseEstimator, TransformerMixin):

    def __init__(self, group_size=10, n_groups=30, top_bins_per_group=3):
        self.group_size = group_size
        self.n_groups = n_groups
        self.top_bins_per_group = top_bins_per_group

    def fit(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y)

        scores, _ = f_classif(X, y)
        scores = np.nan_to_num(scores, nan=0.0, posinf=0.0, neginf=0.0)

        groups = []

        for start in range(0, X.shape[1], self.group_size):
            end = min(start + self.group_size, X.shape[1])
            group_scores = scores[start:end]

            n_top = min(self.top_bins_per_group, len(group_scores))
            score = np.mean(np.sort(group_scores)[-n_top:])

            groups.append((start, end, score))

        groups.sort(key=lambda item: item[2], reverse=True)
        selected_groups = groups[:min(self.n_groups, len(groups))]

        self.selected_regions_ = [
            (start, end, score)
            for start, end, score in selected_groups
        ]

        self.support_ = np.array(
            sorted(
                index
                for start, end, _ in selected_groups
                for index in range(start, end)
            ),
            dtype=int
        )

        self.n_features_in_ = X.shape[1]
        return self

    def transform(self, X):
        return np.asarray(X)[:, self.support_]

    def get_support(self, indices=False):
        if indices:
            return self.support_.copy()

        mask = np.zeros(self.n_features_in_, dtype=bool)
        mask[self.support_] = True
        return mask

## 3. Common ML test: `run_cv_ml` (copied unchanged from Joud's notebook)

The same test as the `run_cv`, extended to any classifier. For one antibiotic:

1. Take only the isolates that have an R/S result.
2. Use the saved folds: train on 4 folds, test on the remaining one, repeated 5 times.
3. In each round, fit the `StandardScaler` and the reducer on the **training folds only**, then train each model on the reduced training data.
4. Score the test fold with **AUC** and **PR-AUC**, and report the mean and spread over the 5 folds.

`get_scores` uses the predicted probability of R when the model has one, otherwise the decision value (for example SVM). AUC and PR-AUC only need the ranking, so both are valid.

`run_all_ml` repeats the test for all 12 antibiotics and returns one table with the columns `antibiotic, input, model, auc, auc_std, prauc, n_features`.

In [6]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

def get_scores(model, X):
    """Score for 'resistant'. AUC and PR-AUC only need the ranking, so any score works."""
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    return model.decision_function(X)          # SVM gives a distance instead of a probability

def run_cv_ml(reducer, models, antibiotic):
    """Same folds as run_cv. Scaler + reducer are fitted ONCE per fold, then every model reuses them."""
    has_label = (lab[antibiotic].notna()) & (lab[f"fold_{antibiotic}"] >= 0)
    Xa = X[has_label.values]
    ya = lab.loc[has_label, antibiotic].astype(int).values
    fa = lab.loc[has_label, f"fold_{antibiotic}"].values

    scores = {name: {"auc": [], "prauc": []} for name in models}
    n_feats = []
    for k in range(5):
        tr, te = fa != k, fa == k
        steps = [StandardScaler()]
        if reducer is not None:
            steps.append(clone(reducer))
        prep = make_pipeline(*steps).fit(Xa[tr], ya[tr])        # fitted on the training folds only
        Ztr, Zte = prep.transform(Xa[tr]), prep.transform(Xa[te])
        n_feats.append(Ztr.shape[1])
        for name, model in models.items():
            m = clone(model).fit(Ztr, ya[tr])                   # fresh copy of the classifier
            p = get_scores(m, Zte)
            scores[name]["auc"].append(roc_auc_score(ya[te], p))
            scores[name]["prauc"].append(average_precision_score(ya[te], p))

    return [{"antibiotic": antibiotic, "model": name,
             "auc": np.mean(s["auc"]), "auc_std": np.std(s["auc"]),
             "prauc": np.mean(s["prauc"]), "n_features": int(np.mean(n_feats))}
            for name, s in scores.items()]

def run_all_ml(input_name, reducer, models, antibiotics=None):
    """Run the models on the 12 antibiotics. Returns one table; nothing is saved here."""
    rows = []
    for abx in (antibiotics or MODEL_ABX):
        for r in run_cv_ml(reducer, models, abx):
            r["input"] = input_name
            rows.append(r)
            print(input_name, "|", r["model"], "|", abx, "| AUC =", round(r["auc"], 3),
                  "| PR-AUC =", round(r["prauc"], 3))
    return pd.DataFrame(rows)[["antibiotic", "input", "model", "auc", "auc_std", "prauc", "n_features"]]

## 4. Check

Before adding new models, we re-run one of Joud's results with this notebook. If the number is identical, the data, folds, reducers and test are identical to hers, and the new results can be merged with `results_ml_1.csv`.

**Test:** SVM (RBF kernel, C = 1) on Kernel PCA RBF-200, Cefepime. Joud's result: **AUC 0.877, PR-AUC 0.519**.

In [7]:
from sklearn.svm import SVC
t = time.time()
check = run_cv_ml(KernelPCAScaled(n_components=200, kernel="rbf"),
                  {"svm": SVC(kernel="rbf", C=1.0, gamma="scale")}, "Cefepime")
print("Time:", round(time.time() - t), "s")
display(pd.DataFrame(check).round(3))

Time: 31 s


,antibiotic,model,auc,auc_std,prauc,n_features
0,Cefepime,svm,0.877,0.02,0.519,200


KSSHIBA needs PyTorch and Pyro: `%pip install torch pyro-ppl`. The cell above is commented out so it does not run again.

In [8]:
# %pip install torch pyro-ppl

## Load the authors' code

Two folders downloaded from GitHub (kept outside the project, not redistributed):
- `RMPrediction-master`: the KSSHIBA model
- `maldi-learn-master`: two helper modules that KSSHIBA imports

The cell finds both folders, adds them to Python's path and loads the model. It also applies **two compatibility fixes**, because the 2022 code uses names removed in newer libraries. They restore the old names and do not change the method:
- `np.NaN`, removed in NumPy 2.0 → set to `np.nan`
- `scipy.linalg.pinv(rcond=...)`, renamed in newer SciPy → mapped to `rtol`

In [9]:
import sys
from pathlib import Path

EXT = Path(r"C:/Users/maria/OneDrive/Desktop/Senior_Project/external")          # the folder where the extracted ZIPs are
print("Folder exists:", EXT.exists())

# Find the two files we need, wherever they are inside EXT
ksshiba_file = next(EXT.rglob("fast_fs_ksshiba_b_ord.py"), None)   # KSSHIBA model
maldi_file   = next(EXT.rglob("maldi_learn/data.py"), None)        # helper module it imports
print("KSSHIBA file:", ksshiba_file)
print("maldi-learn file:", maldi_file)

rm_root    = ksshiba_file.parent.parent          
maldi_root = maldi_file.parent                   
sys.path.insert(0, str(rm_root))
sys.path.insert(0, str(maldi_root))

np.NaN = np.nan                                      # NumPy 2 removed the alias np.NaN
import scipy.linalg as _sla
from scipy.linalg._basic import pinv as _true_pinv   # SciPy's real pinv, never patched
def _pinv_compat(a, rcond=None, **kw):               # newer SciPy renamed pinv(rcond=...) to rtol
    if rcond is not None:
        kw.setdefault("rtol", rcond)
    return _true_pinv(a, **kw)
_sla.pinv = _pinv_compat

from lib import fast_fs_ksshiba_b_ord as ksshiba
print("KSSHIBA loaded")

Folder exists: True
KSSHIBA file: C:\Users\maria\OneDrive\Desktop\Senior_Project\external\RMPrediction-master\lib\fast_fs_ksshiba_b_ord.py
maldi-learn file: C:\Users\maria\OneDrive\Desktop\Senior_Project\external\maldi-learn-master\maldi_learn\data.py
KSSHIBA loaded


## One-fold pilot (Ceftriaxone, fold 0)
Before the full run: check that the model trains on our data and measure the time for one fold. 

In [10]:
import scipy.linalg as _sla
from scipy.linalg._basic import pinv as _true_pinv   # SciPy's real pinv, never patched
def _pinv_compat(a, rcond=None, **kw):
    if rcond is not None:
        kw.setdefault("rtol", rcond)
    return _true_pinv(a, **kw)
_sla.pinv = _pinv_compat
np.NaN = np.nan

abx = "Ceftriaxone"
has = (lab[abx].notna()) & (lab[f"fold_{abx}"] >= 0)
Xa = X[has.values]; ya = lab.loc[has, abx].astype(int).values; fa = lab.loc[has, f"fold_{abx}"].values
tr, te = fa != 0, fa == 0

sc = StandardScaler().fit(Xa[tr])                          # scaler fitted on the training folds only
x_all = np.vstack([sc.transform(Xa[tr]), sc.transform(Xa[te])])   # training spectra first, then test spectra
y_tr = ya[tr].reshape(-1, 1).astype(float)                 # labels of the TRAINING spectra only

t = time.time()
model = ksshiba.SSHIBA(100, 1, fs=1)                       # start with 100 latent factors, prune unused ones
spectra_view = model.struct_data(x_all, method="reg", V=x_all, kernel="linear")  # view 1: kernelized spectra
label_view   = model.struct_data(y_tr, method="mult")      # view 2: R/S label (test labels missing -> predicted)
model.fit(spectra_view, label_view, max_iter=300, pruning_crit=1e-2, verbose=0, feat_crit=1e-2)

p = model.t[1]["mean"][-te.sum():, 0]                      # predicted probability of R for the test spectra
print("KSSHIBA linear, fold 0 | AUC =", round(roc_auc_score(ya[te], p), 3),
      "| PR-AUC =", round(average_precision_score(ya[te], p), 3),
      "| time =", round(time.time() - t), "s")

KSSHIBA linear, fold 0 | AUC = 0.816 | PR-AUC = 0.665 | time = 661 s


## 5. Random Forest

**How it works:** builds 300 decision trees. Each tree is trained on a bootstrap sample of the isolates, and at each split it considers a random subset of the features (√n). The trees vote; the score for R is the share of trees that vote R.
**Settings:** 300 trees, `random_state = 0` (same forest every run), all CPU cores. Default settings first; tuning follows in 5.3.

### 5.1 Quick test (Cefepime, 4 inputs)


In [11]:
from sklearn.ensemble import RandomForestClassifier
import os
print("CPU cores on this laptop:", os.cpu_count())

RF_MODEL = {"rf": RandomForestClassifier(n_estimators=300,    # 300 trees
                                         n_jobs=-1,           # use all CPU cores
                                         random_state=0)}     # same forest every run

INPUTS = {
    "kpca_rbf":      KernelPCAScaled(n_components=200, kernel="rbf"),
    "grouped_anova": GroupedANOVASelector(group_size=10, n_groups=30, top_bins_per_group=3),
    "lasso_autoC":   LassoAutoC(),
    "all_bins":      None,                     # no reduction, all 6,000 bins
}

for input_name, reducer in INPUTS.items():
    t = time.time()
    r = run_cv_ml(reducer, RF_MODEL, "Cefepime")[0]      # Joud's function: same folds, same scaler
    print(f"{input_name:14s} | AUC = {r['auc']:.3f} | PR-AUC = {r['prauc']:.3f} | time = {time.time() - t:.0f} s")

CPU cores on this laptop: 16
kpca_rbf       | AUC = 0.755 | PR-AUC = 0.284 | time = 57 s
grouped_anova  | AUC = 0.864 | PR-AUC = 0.455 | time = 35 s
lasso_autoC    | AUC = 0.861 | PR-AUC = 0.455 | time = 109 s
all_bins       | AUC = 0.826 | PR-AUC = 0.335 | time = 237 s


### 5.2 Random Forest on all 12 antibiotics × 4 inputs (default settings)
Same `run_all_ml` as Joud. Results are saved to `results_ml_member1.csv` after each input; finished inputs are skipped if the cell is run again.

In [ ]:

def run_all_ml(input_name, reducer, models, antibiotics=None):      # same as Joud's version
    rows = []
    for abx in (antibiotics or MODEL_ABX):
        for r in run_cv_ml(reducer, models, abx):
            r["input"] = input_name
            rows.append(r)
            print(input_name, "|", r["model"], "|", abx, "| AUC =", round(r["auc"], 3),
                  "| PR-AUC =", round(r["prauc"], 3))
    return pd.DataFrame(rows)[["antibiotic", "input", "model", "auc", "auc_std", "prauc", "n_features"]]

OUT = "results_ml_member1.csv"
all_rf = pd.read_csv(OUT) if os.path.exists(OUT) else pd.DataFrame()

for input_name, reducer in INPUTS.items():
    if len(all_rf) and ((all_rf["input"] == input_name) & (all_rf["model"] == "rf")).any():
        print(input_name, "already done, skipped")
        continue
    t = time.time()
    res = run_all_ml(input_name, reducer, RF_MODEL)
    all_rf = pd.concat([all_rf, res], ignore_index=True)
    all_rf.to_csv(OUT, index=False)                      # saved after each input
    print(input_name, "finished in", round((time.time() - t) / 60, 1), "min\n")

# Mean over the 12 antibiotics
display(all_rf.groupby(["input", "model"])[["auc", "prauc"]].mean().round(3))

kpca_rbf | rf | Ampicillin-Amoxicillin | AUC = 0.695 | PR-AUC = 0.775
kpca_rbf | rf | Cotrimoxazole | AUC = 0.693 | PR-AUC = 0.526
kpca_rbf | rf | Ciprofloxacin | AUC = 0.781 | PR-AUC = 0.672
kpca_rbf | rf | Levofloxacin | AUC = 0.788 | PR-AUC = 0.678
kpca_rbf | rf | Amoxicillin-Clavulanic acid | AUC = 0.644 | PR-AUC = 0.408
kpca_rbf | rf | Ceftriaxone | AUC = 0.824 | PR-AUC = 0.672
kpca_rbf | rf | Cefpodoxime | AUC = 0.817 | PR-AUC = 0.81
kpca_rbf | rf | Tobramycin | AUC = 0.664 | PR-AUC = 0.273
kpca_rbf | rf | Ceftazidime | AUC = 0.773 | PR-AUC = 0.396
kpca_rbf | rf | Norfloxacin | AUC = 0.697 | PR-AUC = 0.515
kpca_rbf | rf | Cefepime | AUC = 0.755 | PR-AUC = 0.284
kpca_rbf | rf | Piperacillin-Tazobactam | AUC = 0.592 | PR-AUC = 0.143
kpca_rbf finished in 12.5 min

grouped_anova | rf | Ampicillin-Amoxicillin | AUC = 0.722 | PR-AUC = 0.804
grouped_anova | rf | Cotrimoxazole | AUC = 0.724 | PR-AUC = 0.561
grouped_anova | rf | Ciprofloxacin | AUC = 0.824 | PR-AUC = 0.721
grouped_anova |

,,auc,prauc
input,model,,
all_bins,rf,0.761,0.543
grouped_anova,rf,0.777,0.579
kpca_rbf,rf,0.727,0.513
lasso_autoC,rf,0.778,0.582


**Results (mean over 12 antibiotics):**

| Input | LR (reference) | RF | Gradient boosting | SVM | MLP |
|---|---|---|---|---|---|
| All 6,000 bins | 0.744 | 0.761 | 0.793 | 0.796 | 0.768 |
| Grouped ANOVA 300 | 0.755 | 0.777 | 0.783 | 0.792 | 0.768 |
| Kernel PCA RBF-200 | 0.772 | 0.727 | 0.771 | 0.791 | 0.768 |
| LASSO auto C | 0.756 | 0.778 | 0.784 | 0.783 | 0.761 |

(Gradient boosting, SVM and MLP: Joud's untuned results, `results_ml_1.csv`.)

**What we observed**
- **RF beats logistic regression on real bins** (0.761–0.778 vs 0.744–0.756), but it is the weakest of the non-linear models.
- **RF works best on selected bins** (LASSO 0.778, ANOVA 0.777). With all 6,000 bins, most of the √6000 ≈ 77 bins considered at each split are uninformative.
- **RF is poor on Kernel PCA** (0.727, below logistic regression). Trees split on one feature at a time, while Kernel PCA components are mixtures of all bins, so the signal is not aligned with single features.

**Limits:** default settings only (tuning in 5.3); one hospital.

### 5.3 Random Forest tuned 

Same models, but now Random Forest picks its own setting. Inside each training fold, a 3-fold validation tries the grid below and keeps the best by AUC (Joud's `tuned()` function). The test fold is never used for tuning, so there is no leakage.

| Model | Setting tried |
|---|---|
| Random Forest | features considered at each split: √n or log₂n (`max_features` = "sqrt", "log2") |

Run on all 4 inputs, so before and after tuning can be compared for each input. Results are added to `results_ml_member1.csv` with the model name `rf_tuned`. 

In [22]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

def tuned(model, grid):
    """Same as Joud's tuned(): 3-fold validation inside the TRAINING folds only, best by AUC.
    n_jobs is left at default here because the forest itself already uses all CPU cores."""
    return GridSearchCV(model, grid, cv=3, scoring="roc_auc")

RF_TUNED = {"rf_tuned": tuned(RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=0),
                              {"max_features": ["sqrt", "log2"]})}

INPUTS = {
    "kpca_rbf":      KernelPCAScaled(n_components=200, kernel="rbf"),
    "grouped_anova": GroupedANOVASelector(group_size=10, n_groups=30, top_bins_per_group=3),
    "lasso_autoC":   LassoAutoC(),
    "all_bins":      None,
}

OUT = "results_ml_member1.csv"
all_res = pd.read_csv(OUT)
for input_name, reducer in INPUTS.items():
    if ((all_res["input"] == input_name) & (all_res["model"] == "rf_tuned")).any():
        print(input_name, "already done, skipped")
        continue
    t = time.time()
    res = run_all_ml(input_name, reducer, RF_TUNED)
    all_res = pd.concat([all_res, res], ignore_index=True)
    all_res.to_csv(OUT, index=False)                         # saved after each input
    print(input_name, "finished in", round((time.time() - t) / 60, 1), "min\n")

# Before vs after tuning: mean AUC over the 12 antibiotics (same layout as Joud's table)
m = all_res[all_res.model.isin(["rf", "rf_tuned"])].groupby(["input", "model"]).auc.mean().unstack()
m["change"] = m["rf_tuned"] - m["rf"]
display(m.round(3))

kpca_rbf | rf_tuned | Ampicillin-Amoxicillin | AUC = 0.693 | PR-AUC = 0.773
kpca_rbf | rf_tuned | Cotrimoxazole | AUC = 0.693 | PR-AUC = 0.526
kpca_rbf | rf_tuned | Ciprofloxacin | AUC = 0.781 | PR-AUC = 0.672
kpca_rbf | rf_tuned | Levofloxacin | AUC = 0.788 | PR-AUC = 0.678
kpca_rbf | rf_tuned | Amoxicillin-Clavulanic acid | AUC = 0.644 | PR-AUC = 0.409
kpca_rbf | rf_tuned | Ceftriaxone | AUC = 0.824 | PR-AUC = 0.672
kpca_rbf | rf_tuned | Cefpodoxime | AUC = 0.817 | PR-AUC = 0.81
kpca_rbf | rf_tuned | Tobramycin | AUC = 0.662 | PR-AUC = 0.272
kpca_rbf | rf_tuned | Ceftazidime | AUC = 0.777 | PR-AUC = 0.395
kpca_rbf | rf_tuned | Norfloxacin | AUC = 0.698 | PR-AUC = 0.52
kpca_rbf | rf_tuned | Cefepime | AUC = 0.755 | PR-AUC = 0.296
kpca_rbf | rf_tuned | Piperacillin-Tazobactam | AUC = 0.583 | PR-AUC = 0.139
kpca_rbf finished in 19.2 min

grouped_anova | rf_tuned | Ampicillin-Amoxicillin | AUC = 0.722 | PR-AUC = 0.804
grouped_anova | rf_tuned | Cotrimoxazole | AUC = 0.724 | PR-AUC = 0.56

model,rf,rf_tuned,change
input,,,
all_bins,0.761,0.761,0.000
grouped_anova,0.777,0.778,0.001
kpca_rbf,0.727,0.726,-0.001
lasso_autoC,0.778,0.779,0.000


## 6. KSSHIBA (Kernelized Sparse Semi-supervised Heterogeneous Interbattery Bayesian Analysis)

**Source:** Guerrero-López et al. (2023), *Engineering Applications of Artificial Intelligence* 118, 105644. Authors' implementation used unchanged: github.com/aguerrerolopez/RMPrediction (`lib/fast_fs_ksshiba_b_ord.py`).

**How it works:** two views of the same isolates share one low-dimensional latent space:
1. **Spectra view:** the 6,000 bins turned into an N × N similarity table (kernel), as in Eq. 1 of the paper.
2. **Label view:** R/S, modelled with a Bernoulli (sigmoid) link.

The model is trained by variational Bayesian inference (maximising the ELBO). It starts with 100 latent factors and prunes unused ones; there is no hyperparameter search. The test isolates are included with their **labels missing**, and the model predicts them (semi-supervised, as in the paper).

**Kernel: linear.** It is the only kernel for which the paper's Eq. 10 (W = XᵀA) gives an exact weight per bin.

**Input:** all 6,000 bins only. KSSHIBA performs its own reduction (kernel and latent space), so a second reducer is not used.

**Note for the report:** KSSHIBA sees the test *spectra* (never their labels) during training. The other models do not see test spectra at all.

In [ ]:
import scipy.linalg as _sla
from scipy.linalg._basic import pinv as _true_pinv
def _pinv_compat(a, rcond=None, **kw):
    if rcond is not None:
        kw.setdefault("rtol", rcond)
    return _true_pinv(a, **kw)
_sla.pinv = _pinv_compat
np.NaN = np.nan

KS_MAX_IT = 300        # upper limit; the model stops earlier when its ELBO converges (about 170-200 iterations)
KS_SEED = 0            # fixed random start -> the same result every run

def ksshiba_cv(antibiotic):
    """Same 5 folds as everyone. Per fold: KSSHIBA (linear kernel) trained with the training labels only;
    the test labels are left missing and predicted. Also returns each bin's weight on the label (Eq. 10)."""
    has_label = (lab[antibiotic].notna()) & (lab[f"fold_{antibiotic}"] >= 0)
    Xa = X[has_label.values]; ya = lab.loc[has_label, antibiotic].astype(int).values
    fa = lab.loc[has_label, f"fold_{antibiotic}"].values
    aucs, praucs, iters, effs = [], [], [], []
    for f in range(5):
        tr, te = fa != f, fa == f
        sc = StandardScaler().fit(Xa[tr])                                   # training folds only
        x_all = np.vstack([sc.transform(Xa[tr]), sc.transform(Xa[te])])    # training spectra, then test spectra
        y_tr = ya[tr].reshape(-1, 1).astype(float)                          # training labels only
        np.random.seed(KS_SEED)
        model = ksshiba.SSHIBA(100, 1, fs=1)
        model.fit(model.struct_data(x_all, method="reg", V=x_all, kernel="linear"),
                  model.struct_data(y_tr, method="mult"),
                  max_iter=KS_MAX_IT, pruning_crit=1e-2, verbose=0, feat_crit=1e-2)
        p = model.t[1]["mean"][-te.sum():, 0]                              # predicted probability of R (test)
        aucs.append(roc_auc_score(ya[te], p)); praucs.append(average_precision_score(ya[te], p))
        iters.append(len(model.L))
        # Bin weights (Eq. 10): W = X^T A (spectra view), then through the latent space to the label view
        W_bins = x_all.T @ model.q_dist.W[0]["mean"]                       # 6000 x K
        eff = (W_bins @ model.q_dist.W[1]["mean"].T)[:, 0]                  # 6000: >0 = pushes towards R
        effs.append(eff / np.abs(eff).max())                               # scale each fold to [-1, 1]
        print(f"   {antibiotic} fold {f}: AUC {aucs[-1]:.3f} | iterations {iters[-1]}", flush=True)
    return {"antibiotic": antibiotic, "input": "all_bins", "model": "ksshiba_linear",
            "auc": np.mean(aucs), "auc_std": np.std(aucs), "prauc": np.mean(praucs),
            "n_features": X.shape[1], "mean_iterations": np.mean(iters)}, np.array(effs)

OUT = "results_ml_member1.csv"
all_res = pd.read_csv(OUT) if os.path.exists(OUT) else pd.DataFrame()
os.makedirs("ksshiba_bins", exist_ok=True)

for abx in MODEL_ABX:
    if len(all_res) and ((all_res["model"] == "ksshiba_linear") & (all_res["antibiotic"] == abx)).any():
        print(abx, "already done, skipped")
        continue
    t = time.time()
    row, effs = ksshiba_cv(abx)
    np.save(f"ksshiba_bins/ksshiba_weights_{abx}.npy", effs)              # 5 folds x 6000 bin weights
    all_res = pd.concat([all_res, pd.DataFrame([row])], ignore_index=True)
    all_res.to_csv(OUT, index=False)                                      # same file as Random Forest
    print(f"{abx:28s} | AUC {row['auc']:.3f} | PR-AUC {row['prauc']:.3f} | {(time.time() - t) / 60:.1f} min\n", flush=True)

ks = all_res[all_res.model == "ksshiba_linear"]
display(ks[["antibiotic", "auc", "auc_std", "prauc", "mean_iterations"]].round(3))
print("KSSHIBA linear mean AUC:", round(ks.auc.mean(), 3), "| mean PR-AUC:", round(ks.prauc.mean(), 3))

   Ampicillin-Amoxicillin fold 0: AUC 0.658 | iterations 186
   Ampicillin-Amoxicillin fold 1: AUC 0.648 | iterations 179
   Ampicillin-Amoxicillin fold 2: AUC 0.653 | iterations 177
   Ampicillin-Amoxicillin fold 3: AUC 0.680 | iterations 181
   Ampicillin-Amoxicillin fold 4: AUC 0.638 | iterations 178
Ampicillin-Amoxicillin       | AUC 0.656 | PR-AUC 0.752 | 25.4 min

   Cotrimoxazole fold 0: AUC 0.683 | iterations 177
   Cotrimoxazole fold 1: AUC 0.657 | iterations 179
   Cotrimoxazole fold 2: AUC 0.649 | iterations 180
   Cotrimoxazole fold 3: AUC 0.662 | iterations 176
   Cotrimoxazole fold 4: AUC 0.636 | iterations 176
Cotrimoxazole                | AUC 0.658 | PR-AUC 0.460 | 25.5 min

   Ciprofloxacin fold 0: AUC 0.812 | iterations 171
   Ciprofloxacin fold 1: AUC 0.802 | iterations 179
   Ciprofloxacin fold 2: AUC 0.757 | iterations 176
   Ciprofloxacin fold 3: AUC 0.789 | iterations 180
   Ciprofloxacin fold 4: AUC 0.766 | iterations 178
Ciprofloxacin                | AUC 0.78

,antibiotic,auc,auc_std,prauc,mean_iterations
48,Ampicillin-Amoxicillin,0.656,0.014,0.752,180.2
49,Cotrimoxazole,0.658,0.015,0.460,177.6
50,Ciprofloxacin,0.785,0.021,0.664,176.8
51,Levofloxacin,0.786,0.027,0.670,177.2
52,Amoxicillin-Clavulanic acid,0.624,0.035,0.373,179.0
53,Ceftriaxone,0.824,0.009,0.655,175.4
54,Cefpodoxime,0.810,0.015,0.792,199.6
55,Tobramycin,0.686,0.013,0.255,178.6
56,Ceftazidime,0.778,0.010,0.368,180.2
57,Norfloxacin,0.681,0.043,0.451,190.6


KSSHIBA linear mean AUC: 0.724 | mean PR-AUC: 0.486


**Results:** mean AUC **0.724**, mean PR-AUC **0.486** (logistic regression on all bins: 0.744 / 0.509; Random Forest 0.761; SVM 0.796). Every antibiotic converged after 175–200 iterations.

**What we observed**
- **KSSHIBA ≈ logistic regression where resistance is common** and data is plentiful (Ampicillin-Amoxicillin, Cotrimoxazole, Ciprofloxacin, Levofloxacin, Amoxicillin-Clavulanic acid: −0.001 to +0.017). This is expected: with a linear kernel, KSSHIBA is a Bayesian *linear* model.
- **It is clearly worse on rare resistance:** Cefepime (7.6% R, −0.054) and Piperacillin-Tazobactam (6.4% R, −0.074). Across the 12 antibiotics, the loss grows as the share of R falls (correlation 0.51). The Bernoulli label view has no class weighting.
- **It is worse on the smallest set:** Norfloxacin, 1,470 isolates, −0.078.
- **No tuning** was needed or done: an advantage in effort, but here it did not beat the tuned models.

**Limits:** one hospital, so the paper's main strength (a domain view to combine hospitals) was not used; linear kernel only; the test spectra were seen during training (semi-supervised design).

## 7. All ML models in one table 

gradient boosting, SVM, MLP, before and after tuning, Random Forest before and after tuning, KSSHIBA are merged into `results_ml_all_models.csv`. All rows use the same data, folds, scaler, inputs and test, so they are directly comparable.
KSSHIBA has one input only (all bins), so its other cells are empty.

In [25]:
mine = pd.read_csv("results_ml_member1.csv").drop(columns="mean_iterations", errors="ignore")
joud = pd.read_csv("results_ml_1 (1).csv")
all_ml = pd.concat([joud, mine], ignore_index=True)
all_ml.to_csv("results_ml_all_models.csv", index=False)          # merged file

order = [m for m in ["gboost", "gboost_tuned", "svm", "svm_tuned", "mlp", "mlp_tuned",
                     "rf", "rf_tuned", "ksshiba_linear"] if m in set(all_ml.model)]
print("Mean AUC over the 12 antibiotics (rows = input, columns = model)")
display(all_ml.groupby(["input", "model"]).auc.mean().unstack()[order].round(3))
print("Mean PR-AUC over the 12 antibiotics")
display(all_ml.groupby(["input", "model"]).prauc.mean().unstack()[order].round(3))

Mean AUC over the 12 antibiotics (rows = input, columns = model)


model,gboost,gboost_tuned,svm,svm_tuned,mlp,mlp_tuned,rf,rf_tuned,ksshiba_linear
input,,,,,,,,,
all_bins,0.793,0.789,0.796,0.807,0.768,0.765,0.761,0.761,0.724
grouped_anova,0.783,0.781,0.792,0.795,0.768,0.769,0.777,0.778,NaN
kpca_rbf,0.771,0.767,0.791,0.805,0.768,0.772,0.727,0.726,NaN
lasso_autoC,0.784,0.784,0.783,0.783,0.761,0.763,0.778,0.779,NaN


Mean PR-AUC over the 12 antibiotics


model,gboost,gboost_tuned,svm,svm_tuned,mlp,mlp_tuned,rf,rf_tuned,ksshiba_linear
input,,,,,,,,,
all_bins,0.600,0.591,0.621,0.627,0.548,0.551,0.543,0.543,0.486
grouped_anova,0.589,0.581,0.614,0.616,0.567,0.568,0.579,0.579,NaN
kpca_rbf,0.564,0.556,0.612,0.628,0.566,0.567,0.513,0.513,NaN
lasso_autoC,0.584,0.585,0.594,0.588,0.553,0.556,0.582,0.582,NaN
